# Step-by-step demo of NEDAS using the vort2d model



## Setup
NEDAS is available from the PyPI platform. You can install with ```pip install NEDAS```.

You can also clone the `NEDAS` repository and install in editable mode ```cd NEDAS; pip install -e .``` for active code development.

In [ ]:
# only need to do this once:
to_install = False

if to_install:
    # 1. Install the latest version of NEDAS on develop branch
    %cd ~
    %rm -rf NEDAS
    !git clone https://github.com/nansencenter/NEDAS.git
    %cd NEDAS
    %pip install .
    
    # 2. Install additional dependencies
    # numba provides JIT compilation of python function to speed it up during runtime
    %pip install numba
    # cmocean provides better colormaps for visualization
    %pip install cmocean
    
    # 3. If in Google Colab, need to clone the tutorial repo too
    import sys
    if 'google.colab' in sys.modules:
        %mkdir ~/work
        %cd ~/work
        %rm -rf NEDAS_tutorials
        !git clone https://github.com/myying/NEDAS_tutorials.git

%cd ~/work/NEDAS_tutorials


In [ ]:
# utility modules
import os
import numpy as np
from datetime import datetime, timedelta

# a few graphic modules
import matplotlib.pyplot as plt
import cmocean
from matplotlib import cm
from matplotlib.animation import FuncAnimation
from IPython.core.display import HTML


Check how many CPUs are available on your system. You can then set `nproc`, `nproc_util` in the configuration accordingly

**Notes**:
- If `mpi4py` is not supported, you shall use `nproc=1`
- For the vort2d model with relatively small dimension, `nproc=1` is already fast and increasing it won't speed up due to additional overheads

In [ ]:
os.cpu_count()

In [ ]:
# this shell command checks if mpi4py is working normally
import subprocess
subprocess.run("mpirun -n 4 python -c 'from mpi4py import MPI; print(MPI.COMM_WORLD.Get_rank())'", shell=True)

## Prerequisites

### Configuration

A YAML file `vort2d/config.yml` contains all the settings for an experiment.

See [Configuration file](https://nedas.readthedocs.io/en/latest/config_file.html) documentation for more details.

In command line, you can run the experiment with `python -m NEDAS -c vort2d/config.yml`

Here we setup the `config` object in an interactive environment:

In [ ]:
from NEDAS.config import Config

# load configuration YAML file
# additional kwargs will overwrite the values in the file
config = Config(config_file="vort2d/config.yml", nproc=1, quiet=False)

# you can also change values like this
config.debug = False
config.io_mode = 'online'
config.cycle_period = 6
config.model_def['vort2d']['dt'] = 60

# to list all configuration variables
config.__dict__

In [ ]:
# Once you are happy with the configuration,
# you can initialize the main Scheme object
from NEDAS import get_scheme
scheme = get_scheme(config)

In [ ]:
# For convenience, we make alias of a few frequently used objects

# model object, which is a Vort2DModel(Model) instance
model = scheme.c.models['vort2d']
from NEDAS.models.vort2d import Vort2DModel
assert isinstance(model, Vort2DModel)

# dataset object, which is a Vort2DObs(SyntheticObs) instance
dataset = scheme.c.datasets['vort2d']
from NEDAS.datasets.vort2d import Vort2DObs
assert isinstance(dataset, Vort2DObs)

# and the analysis grid (same as model grid), which is a RegularGrid instance
from NEDAS.grid import RegularGrid
grid = scheme.c.grid


In [ ]:
# clear previous results
%rm -rf vort2d/work

model.memory = {}
dataset.memory = {}

### Some diagnostic tools

To visualize the state of the vort2d model, it 

Diagnostics to measure the performance, in both physical and spectral spaces

- root-mean-square error (RMSE)
- ensemble spread

Visualization

- vorticity maps
- spectrum
- sawtooth time series

In [ ]:
def get_model_state(c, t, m):
    if t == c.time:
        try:
            state = c.io.call_method(c, 'post', model.read_var, name='velocity', member=m, time=t, model_src='vort2d')
        except:
            state = c.io.call_method(c, 'current', model.read_var, name='velocity', member=m, time=t, model_src='vort2d')
    else:
        try:
            state = c.io.call_method(c, 'prior', model.read_var, name='velocity', member=m, time=t, model_src='vort2d')
        except:
            state = c.io.call_method(c, 'current', model.read_var, name='velocity', member=m, time=t, model_src='vort2d')
    return state

In [ ]:
from NEDAS.utils.spatial_operation import gradx, grady

# compute vorticity (1/s) from velocity (m/s)
def uv2zeta(vel):
    u, v = vel[0], vel[1]
    zeta = gradx(v, grid.dx, grid.cyclic_dim) - grady(u, grid.dy, grid.cyclic_dim)
    return zeta

# color map and range of vorticity to show (1/s)
vort_cmap = getattr(cmocean.cm, 'curl')
vort_min = -5e-3
vort_max = 5e-3
vort_intv = 1e-3

In [ ]:
# domain-averaged RMSE
def rmse(Xens, Xt):
    return np.sqrt(np.mean((np.mean(Xens, axis=1)-Xt)**2, axis=(1,2,3)))

# domain-averaged ensemble spread
def sprd(Xens):
    return np.sqrt(np.mean(np.std(Xens, axis=1)**2, axis=(1,2,3)))

# spectrum of error and ensemble spread
from NEDAS.diag.metrics.spectral import pwrspec2d
from NEDAS.utils.fft_lib import get_wn

def err_spec(Xens, Xt):
    wn, err_pwr = pwrspec2d(np.mean(Xens, axis=0)-Xt)
    return wn, err_pwr

def sprd_spec(Xens):
    nens, nv, nj, ni = Xens.shape
    ki, kj = get_wn(Xens[-2:])
    nup = int(max(ki.max(), kj.max()))
    Xmean = np.mean(Xens, axis=0)
    pwr_ens = np.zeros((nens, nv, nup))
    for m in range(nens):
        wn, pwr_ens[m, ...] = pwrspec2d(Xens[m, ...] - Xmean)
    sprd_pwr = np.sum(pwr_ens, axis=0) / (nens-1)
    return wn, sprd_pwr

In [ ]:
# true vorticity map, highlighted contour in black, and ensemble members in colors
def plot_vorticity_map(ax, n, ts, Xt, Xens):
    ax.clear()
    vort_truth = uv2zeta(Xt[n])
    vort_truth[np.where(vort_truth>vort_max)] = vort_max
    vort_truth[np.where(vort_truth<vort_min)] = vort_min
    ax.contourf(vort_truth, np.arange(vort_min, vort_max+vort_intv, vort_intv), cmap=vort_cmap)
    clvs = [-1e-3, 1e-3]
    cmap = [plt.cm.jet(x) for x in np.linspace(0, 1, scheme.c.nens)]
    for m in range(scheme.c.nens):
        vort_mem = uv2zeta(Xens[n,m])
        ax.contour(vort_mem, clvs, colors=[cmap[m][0:3]], linewidths=1)
    ax.contour(vort_truth, clvs, colors='k', linewidths=2)
    ax.set_title(fr"vorticity $t$={ts[n]:02}h")
    cticks = np.array([5, 15, 25, 35, 45])
    ax.set_xticks(cticks)
    ax.set_xticklabels(((grid.x[0,cticks] - grid.Lx/2)/1e3).astype(int))
    ax.set_yticks(cticks)
    ax.set_yticklabels(((grid.y[cticks,0] - grid.Ly/2)/1e3).astype(int))
    ax.set_xlabel(r'$x$ (km)')
    ax.set_ylabel(r'$y$ (km)')

# power spectra of error and ensemble spread
def plot_spectrum(ax, n, ts, Xt, Xens):
    ax.clear()
    wn, pwr = pwrspec2d(Xt[n])
    ax.loglog(wn, np.mean(pwr, axis=0), color='k', linewidth=2, label='truth')
    wn, err_pwr = err_spec(Xens[n], Xt[n])
    ax.loglog(wn, np.mean(err_pwr, axis=0), color='g', linewidth=3, label='rmse')
    wn, sprd_pwr = sprd_spec(Xens[n])
    ax.loglog(wn, np.mean(sprd_pwr, axis=0), color='r', linestyle=':', linewidth=2, label='sprd')
    ax.legend()
    ax.set_title(fr'spectrum $t$={ts[n]:02}h')
    ax.set_xlim([0.8, 12])
    ax.set_ylim([1e-5, 1e2])
    lengths = np.array([500, 200, 100, 50, 20])
    ax.set_xticks(grid.Lx / 1e3 / lengths)
    ax.set_xticklabels(lengths)
    ax.set_xlabel('wavelength (km)')
    ax.set_ylabel(r'$m^2/s^2$')
    ax.grid()

# Sawtooth time series of error and ensemble spread
def plot_sawtooth_ts(ax, n, ts, rmse_ts, sprd_ts):
    dt = model.restart_dt
    ax.clear()
    ax.plot(ts[0:n+1], rmse_ts[0:n+1], color='g', linewidth=3)
    ax.plot(ts[0:n+1], sprd_ts[0:n+1], color='r', linestyle=':', linewidth=2)
    ax.plot(ts[n], rmse_ts[n], color='k', marker='+', markersize=10)
    ax.set_title('domain-avg rmse,sprd')
    ax.set_xlim([-dt, np.max(ts)+dt])
    ax.set_xticks([0, 12, 24, 36, 48])
    ax.set_ylim([0, 20])
    ax.set_xlabel(r'forecast $t$ (h)')
    ax.set_ylabel(r'$m/s$')
    ax.grid()

### The verifying truth

We will run an Observing System Simulation Experiment (OSSE) for the vort2d model.

First step is to generate a "verifying truth", or a "nature run", which will be used both for generating synthetic observations and for verification of the results.

In [ ]:
# set time to the start of the experiment
scheme.c.time = config.time_start

# for the truth, we place the vortex in the center of the domain (without perturbing its position)
model.loc_sprd = 0

# run the model from time_start to time_end and save results
# in offline IO mode, the truth states are saved under model.truth_dir (vort2d/work/truth/*nc files)
scheme.run_step('prepare_truth')

In [ ]:
scheme.c.time = config.time_start
truth_state = []
times = []
while scheme.c.time < config.time_end:
    t = scheme.c.time
    while t <= scheme.c.next_time:
        times.append(t)
        truth = scheme.c.io.call_method(scheme.c, 'truth', model.read_var, name='velocity', member=None, time=t, model_src='vort2d')
        truth_state.append(truth)
        t += model.restart_dt * timedelta(hours=1)
    scheme.c.time = scheme.c.next_time

### Generation of initial ensemble

Second,

In [ ]:
scheme.c.time = config.time_start
model.loc_sprd = 5000
scheme.run_step('prepare_init_ensemble')

model.save_memory('current', time=scheme.c.time)

### Ensemble forecast

In [ ]:
scheme.c.progress.call_stack = []
scheme.c.progress.push('')
model.load_memory('current', scheme.c.time)

scheme.c.time = config.time_start
scheme.c.log_event("CYCLING START...")
while scheme.c.time < config.time_end:
    scheme.c.log_event(f"CURRENT CYCLE: {scheme.c.time} -> {scheme.c.next_time}")
    scheme.run_step('preprocess')
    scheme.run_step('postprocess')
    scheme.run_step('ensemble_forecast')
    scheme.c.time = scheme.c.next_time
scheme.c.log_event("CYCLING COMPLETE.", flag='finish')

In [ ]:
scheme.c.time = config.time_start
ens_state = []
while scheme.c.time < config.time_end:
    t = scheme.c.time
    while t <= scheme.c.next_time:
        ens = []
        for m in range(scheme.c.nens):
            ens.append(get_model_state(scheme.c, t, m))
        ens_state.append(ens)
        t += model.restart_dt * timedelta(hours=1)

    scheme.c.time = scheme.c.next_time

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(9, 3), constrained_layout=True)

nt = len(truth_state)
Xt = np.array(truth_state)
Xens = np.array(ens_state)
dt = model.restart_dt * timedelta(hours=1)
ts = [int((t - times[0]) / dt) for t in times]
rmse_ts = rmse(Xens, Xt)
sprd_ts = sprd(Xens)

def plot_frame(n):
    plot_vorticity_map(ax[0], n, ts, Xt, Xens)
    plot_spectrum(ax[1], n, ts, Xt, Xens)
    plot_sawtooth_ts(ax[2], n, ts, rmse_ts, sprd_ts)

plt.close()
ts_out = [times.index(t) for t in np.unique(times)]
ani = FuncAnimation(fig, plot_frame, frames=ts_out, interval=300)
HTML(ani.to_jshtml())

## Data assimilation

### Generation of synthetic observations

In [ ]:
scheme.c.time = config.time_start

### Assimilating a single observation

### Assimilating a full observation network

## Running experiments

Running the entire scheme, cycling from time_start to time_end

A few things to play with

Since the step by step shown, only works with nproc=1

If you will try nproc>1, use scheme() directly

In [ ]:
scheme.c.time = config.time_start
scheme.c.progress.call_stack = []
scheme()

In [ ]:
scheme.c.time = config.time_start

ens_state = []
while scheme.c.time < config.time_end:
    t = scheme.c.time
    while t <= scheme.c.next_time:
        ens  = []
        for m in range(scheme.c.nens):
            ens.append(get_model_state(scheme.c, t, m))
        ens_state.append(ens)
        t += timedelta(hours=1)

    scheme.c.time = scheme.c.next_time


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(9, 3), constrained_layout=True)

nt = len(truth_state)
Xt = np.array(truth_state)
Xens = np.array(ens_state)
dt = model.restart_dt * timedelta(hours=1)
ts = [int((t - times[0]) / dt) for t in times]
rmse_ts = rmse(Xens, Xt)
sprd_ts = sprd(Xens)

def plot_frame(n):
    plot_vorticity_map(ax[0], n, ts, Xt, Xens)
    plot_spectrum(ax[1], n, ts, Xt, Xens)
    plot_sawtooth_ts(ax[2], n, ts, rmse_ts, sprd_ts)

plt.close()
ani = FuncAnimation(fig, plot_frame, frames=range(nt), interval=300)
HTML(ani.to_jshtml())
